# Notebook 5 – Data Validation

Validating the `customer_transactions_raw.csv` dataset against expected ranges, types, formats, categories, business rules, referential integrity, dates, and general constraints.

In [34]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df.shape

(1000, 12)

In [35]:
df.head()

,customer_id,age,gender,annual_income,city,membership_type,purchase_amount,quantity,signup_date,payment_method,rating,notes
0,100508,43.0,female,80242.98,Bengaluru,Gold,99.13,1.0,08-22-2021,Debit Card,1,NaN
1,100819,41.0,Female,NaN,Delhi,Silver,113.23,1.0,16 Jun 2020,Net Banking,4,NaN
2,100453,61.0,Male,56285.40,Bengaluru,Gold,248.19,2.0,28 Sep 2020,Debit Card,4,NaN
3,100369,NaN,Male,81878.74,NaN,Silver,51.27,9.0,10 Sep 2023,Credit Card,4,NaN
4,100243,24.0,Male,100566.42,Delhi,Silver,97.97,1.0,07/01/2021,Credit Card,4,NaN


## 1. What is Data Validation?

Data validation is the process of checking that data meets defined rules for correctness, consistency, and quality before it is used for analysis or decision-making. It catches issues such as impossible values, wrong data types, badly formatted entries, and violations of business logic.

## 2. Range Validation

Range validation checks that numeric values fall within a logically acceptable range.

**Examples:** Age cannot be negative. Salary cannot be negative. A rating must fall between 0 and 5.

In [36]:
def validate_age_range(age):
    if pd.isna(age):
        return False
    return 0 <= age <= 100
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['valid_age_range'] = df['age_numeric'].apply(validate_age_range)
df['valid_age_range'].value_counts()

valid_age_range
True     940
False     60
Name: count, dtype: int64

In [37]:
def validate_income_range(income):
    if pd.isna(income):
        return False
    return income >= 0
df['valid_income_range'] = df['annual_income'].apply(validate_income_range)
df['valid_income_range'].value_counts()

valid_income_range
True     941
False     59
Name: count, dtype: int64

In [38]:
def validate_rating_range(rating):
    return 0 <= rating <= 5
df['valid_rating_range'] = df['rating'].apply(validate_rating_range)
df['valid_rating_range'].value_counts()

valid_rating_range
True     993
False      7
Name: count, dtype: int64

In [39]:
df[~df['valid_rating_range']][['customer_id', 'rating']]

,customer_id,rating
186,100201,-1
344,100769,-1
620,100862,7
628,100862,7
731,100947,7
858,100563,10
969,100563,10


## 3. Type Validation

Type validation checks that a column's values actually match the expected data type (numeric, string, date, etc.), rather than trusting the column's declared dtype.

In [13]:
def is_valid_number(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False
df['valid_age_type'] = df['age'].apply(is_valid_number)
df['valid_age_type'].value_counts()

valid_age_type
True     991
False      9
Name: count, dtype: int64

In [14]:
def is_valid_amount(value):
    if pd.isna(value):
        return False
    cleaned = str(value).replace('$', '').strip()
    try:
        float(cleaned)
        return True
    except ValueError:
        return False
df['valid_purchase_amount_type'] = df['purchase_amount'].apply(is_valid_amount)
df['valid_purchase_amount_type'].value_counts()

valid_purchase_amount_type
True    1000
Name: count, dtype: int64

In [16]:
df[~df['valid_purchase_amount_type']][['customer_id', 'purchase_amount']].head()

,customer_id,purchase_amount


## 4. Format Validation

Format validation checks that a value follows an expected pattern or structure, such as a date format, an email address, or a phone number.

In [17]:
import re
def validate_date_format(value, patterns):
    if pd.isna(value):
        return False
    return any(re.fullmatch(pattern, str(value).strip()) for pattern in patterns)
date_patterns = [
    r'\d{2}-\d{2}-\d{4}',
    r'\d{2}/\d{2}/\d{4}',
    r'\d{4}-\d{2}-\d{2}',
    r'\d{2} [A-Za-z]{3} \d{4}',
]
df['valid_date_format'] = df['signup_date'].apply(lambda x: validate_date_format(x, date_patterns))
df['valid_date_format'].value_counts()

valid_date_format
True     984
False     16
Name: count, dtype: int64

In [19]:
def validate_customer_id_format(cid):
    return bool(re.fullmatch(r'100\d{3}', str(cid)))
df['valid_customer_id_format'] = df['customer_id'].apply(validate_customer_id_format)
df['valid_customer_id_format'].value_counts()

valid_customer_id_format
True    1000
Name: count, dtype: int64

## 5. Category Validation

Category validation checks that a column's values belong to an accepted, predefined set of categories.

**Example:** Gender/category must belong to an accepted set.

In [20]:
accepted_genders = {'male', 'female'}
def validate_gender(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in accepted_genders
df['valid_gender'] = df['gender'].apply(validate_gender)
df['valid_gender'].value_counts()

valid_gender
True     774
False    226
Name: count, dtype: int64

In [21]:
accepted_membership = {'bronze', 'silver', 'gold', 'platinum'}

def validate_membership(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in accepted_membership

df['valid_membership'] = df['membership_type'].apply(validate_membership)
df['valid_membership'].value_counts()

valid_membership
True     937
False     63
Name: count, dtype: int64

In [22]:
accepted_payment_methods = {'debit card', 'credit card', 'net banking', 'upi', 'cod'}
def validate_payment_method(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in accepted_payment_methods
df['valid_payment_method'] = df['payment_method'].apply(validate_payment_method)
df['valid_payment_method'].value_counts()

valid_payment_method
True     946
False     54
Name: count, dtype: int64

In [15]:
df[~df['valid_payment_method']][['customer_id', 'payment_method']].drop_duplicates()

,customer_id,payment_method
38,100645,NaN
94,100415,crypto
114,100869,NaN
138,100359,crypto
150,100355,NaN
159,100229,NaN
191,100827,crypto
198,100609,NaN
303,100805,NaN
338,100162,crypto


## 6. Null Validation

Null validation checks which columns should never be empty and flags rows that violate this.

In [23]:
required_columns = ['customer_id', 'age', 'gender', 'annual_income', 'signup_date']
def validate_required_fields(row):
    return all(pd.notna(row[col]) and str(row[col]).strip().lower() != 'nan' for col in required_columns)
df['valid_required_fields'] = df.apply(validate_required_fields, axis=1)
df['valid_required_fields'].value_counts()

valid_required_fields
True     806
False    194
Name: count, dtype: int64

In [17]:
df.isnull().sum()

customer_id                      0
age                             44
gender                          90
annual_income                   57
city                           153
membership_type                 63
purchase_amount                  0
quantity                        20
signup_date                     16
payment_method                  34
rating                           0
notes                         1000
age_numeric                     53
valid_age_range                  0
valid_income_range               0
valid_rating_range               0
valid_age_type                   0
valid_purchase_amount_type       0
valid_date_format                0
valid_customer_id_format         0
valid_gender                     0
valid_membership                 0
valid_payment_method             0
valid_required_fields            0
dtype: int64

## 7. Business Rule Validation

Business rule validation enforces domain-specific logic that goes beyond simple type or range checks.

**Examples:** Percentage must be between 0 and 100. Purchase amount times quantity should make sense against a discount rule.

In [24]:
def validate_percentage(value):
    if pd.isna(value):
        return False
    return 0 <= value <= 100
sample_percentages = pd.Series([25.0, 101.0, -5.0, 87.5, np.nan])
sample_percentages.apply(validate_percentage)

0     True
1    False
2    False
3     True
4    False
dtype: bool

In [25]:
def validate_quantity_business_rule(qty):
    if pd.isna(qty):
        return False
    return 1 <= qty <= 20
df['valid_quantity'] = df['quantity'].apply(validate_quantity_business_rule)
df['valid_quantity'].value_counts()

valid_quantity
True     975
False     25
Name: count, dtype: int64

## 8. Referential Validation

Referential validation checks that a value in one column correctly references a valid value elsewhere, such as a foreign key existing in a lookup table.

In [26]:
valid_cities = {'bengaluru', 'delhi', 'chennai', 'hyderabad', 'mumbai'}
def validate_city_reference(city):
    if pd.isna(city):
        return False
    cleaned = str(city).strip().lower()
    return cleaned in valid_cities
df['valid_city_reference'] = df['city'].apply(validate_city_reference)
df['valid_city_reference'].value_counts()

valid_city_reference
True     847
False    153
Name: count, dtype: int64

In [27]:
df[~df['valid_city_reference']]['city'].unique()

<ArrowStringArray>
[nan]
Length: 1, dtype: str

## 9. Date Validation

Date validation checks that dates are parseable, fall within a reasonable range, and respect logical ordering rules.

**Example:** Date of joining cannot be before date of birth.

In [29]:
def parse_flexible_date(value):
    if pd.isna(value):
        return pd.NaT
    for fmt in ('%m-%d-%Y', '%d/%m/%Y', '%Y-%m-%d', '%d %b %Y'):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT
df['signup_date_parsed'] = df['signup_date'].apply(parse_flexible_date)
df['signup_date_parsed'].isna().sum()

np.int64(16)

In [30]:
def validate_signup_date_range(date, min_date='2015-01-01', max_date='2026-09-01'):
    if pd.isna(date):
        return False
    return pd.Timestamp(min_date) <= date <= pd.Timestamp(max_date)
df['valid_signup_date_range'] = df['signup_date_parsed'].apply(validate_signup_date_range)
df['valid_signup_date_range'].value_counts()

valid_signup_date_range
True     984
False     16
Name: count, dtype: int64

In [31]:
def validate_join_after_birth(date_of_birth, date_of_joining):
    if pd.isna(date_of_birth) or pd.isna(date_of_joining):
        return False
    return date_of_joining >= date_of_birth
sample_dob = pd.to_datetime(['2000-01-01', '1995-06-15', '2010-03-20'])
sample_doj = pd.to_datetime(['2020-05-01', '1990-01-01', '2022-07-01'])
[validate_join_after_birth(dob, doj) for dob, doj in zip(sample_dob, sample_doj)]

[True, False, True]

## 10. Constraint Validation

Constraint validation groups multiple rules together and checks whether a record satisfies all defined constraints at once, similar to how a database enforces CHECK constraints.

In [32]:
def validate_record(row):
    checks = {
        'age_ok': validate_age_range(row['age_numeric']),
        'income_ok': validate_income_range(row['annual_income']),
        'rating_ok': validate_rating_range(row['rating']),
        'gender_ok': validate_gender(row['gender']),
        'city_ok': validate_city_reference(row['city']),
        'date_ok': validate_signup_date_range(row['signup_date_parsed']),
    }
    return all(checks.values())
df['record_is_valid'] = df.apply(validate_record, axis=1)
df['record_is_valid'].value_counts()

record_is_valid
True     564
False    436
Name: count, dtype: int64

In [33]:
validation_summary = pd.DataFrame({
    'rule': ['age_range', 'income_range', 'rating_range', 'gender', 'membership',
             'payment_method', 'required_fields', 'quantity', 'city_reference',
             'signup_date_range', 'overall_record'],
    'pass_count': [
        df['valid_age_range'].sum(),
        df['valid_income_range'].sum(),
        df['valid_rating_range'].sum(),
        df['valid_gender'].sum(),
        df['valid_membership'].sum(),
        df['valid_payment_method'].sum(),
        df['valid_required_fields'].sum(),
        df['valid_quantity'].sum(),
        df['valid_city_reference'].sum(),
        df['valid_signup_date_range'].sum(),
        df['record_is_valid'].sum(),
    ]
})
validation_summary['fail_count'] = len(df) - validation_summary['pass_count']
validation_summary

,rule,pass_count,fail_count
0,age_range,940,60
1,income_range,941,59
2,rating_range,993,7
3,gender,774,226
4,membership,937,63
5,payment_method,946,54
6,required_fields,806,194
7,quantity,975,25
8,city_reference,847,153
9,signup_date_range,984,16


In [27]:
df.to_csv('customer_transactions_validated.csv', index=False)
df.shape

(1000, 29)